# MotionJSON Colab Provider Diagnostics Notebook

Use this notebook before choosing providers. It reports which no-model, local-model, hosted, detector, runtime, and export capabilities are installed, configured, and runnable in the current Colab runtime.

This notebook is a diagnostics and CPU/no-model smoke path. This notebook intentionally does not request API keys or provider secrets, does not install heavy local SAM packages, and does not claim SAM2/SAM3 can run just because a setting or environment variable exists. Keep credentials, private videos, SAM checkpoints, and downloaded diagnostic outputs out of shared notebooks.

Colab GPU availability, memory, runtime length, and VM lifetime are not guaranteed. Re-run diagnostics in the current runtime before choosing local SAM2, local SAM3, hosted SAM2, or hosted SAM3.


## 1. Clone and install MotionJSON


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
import textwrap
import time
import urllib.error
import urllib.request

REPO_URL = "https://github.com/ptse8204/json-animated-video.git"
REPO_DIR = Path("/content/json-animated-video") if Path("/content").exists() else Path("json-animated-video")

def run(cmd, *, cwd=None, check=True, capture=False):
    """Run a command with readable echoing for Colab and local notebooks."""
    if isinstance(cmd, str):
        display_cmd = cmd
        shell = True
    else:
        display_cmd = " ".join(shlex.quote(str(part)) for part in cmd)
        shell = False
    print(f"$ {display_cmd}")
    completed = subprocess.run(
        cmd,
        cwd=cwd,
        check=check,
        shell=shell,
        text=True,
        capture_output=capture,
    )
    if capture:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr, file=sys.stderr)
    return completed

if not REPO_DIR.exists():
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    print(f"Using existing checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Repository: {Path.cwd()}")
run([sys.executable, "-m", "pip", "install", "-U", "pip"])
run([sys.executable, "-m", "pip", "install", "-e", ".[ui]"])


## 2. Run text diagnostics


In [ ]:
run([sys.executable, "-m", "motionjson.cli", "backend", "diagnostics", "--text"])


## 3. Parse JSON diagnostics into a compact table

The exact diagnostics schema can evolve, so this parser walks the JSON and extracts records that look like provider or capability status objects.


In [ ]:
completed = subprocess.run(
    [sys.executable, "-m", "motionjson.cli", "backend", "diagnostics", "--json"],
    cwd=REPO_DIR,
    check=True,
    text=True,
    capture_output=True,
)
diagnostics = json.loads(completed.stdout)
import re

sensitive_key_fragments = ("key", "token", "secret", "password", "bearer", "authorization")
secret_value_patterns = [
    re.compile(r"(?i)bearer\s+[A-Za-z0-9._~+/-]+=*"),
    re.compile(r"(?i)(api[_-]?key|token|secret|password)\s*[:=]\s*[^\s,;]+"),
    re.compile(r"sk-[A-Za-z0-9_-]{12,}"),
]

def redact_string(text):
    redacted = text
    for pattern in secret_value_patterns:
        redacted = pattern.sub("<redacted>", redacted)
    return redacted

def redact_diagnostics(value, key=""):
    key_lower = key.lower()
    if any(fragment in key_lower for fragment in sensitive_key_fragments):
        if value in (None, "", False):
            return value
        return "<redacted>"
    if isinstance(value, dict):
        return {child_key: redact_diagnostics(child, child_key) for child_key, child in value.items()}
    if isinstance(value, list):
        return [redact_diagnostics(child, key) for child in value]
    if isinstance(value, str):
        return redact_string(value)
    return value

safe_diagnostics = redact_diagnostics(diagnostics)
print(json.dumps(safe_diagnostics, indent=2)[:6000])

records = []
status_keys = {"installed", "configured", "runnable", "ready", "available", "enabled", "ok"}
name_keys = {"provider", "providerName", "id", "name", "capability"}

def walk(value, path="diagnostics"):
    if isinstance(value, dict):
        has_status = any(key in value for key in status_keys)
        has_name = any(key in value for key in name_keys)
        if has_status or has_name:
            row = {"path": path}
            for key in sorted(name_keys | status_keys | {"reason", "message", "hint", "error"}):
                if key in value and not isinstance(value[key], (dict, list)):
                    row[key] = value[key]
            records.append(row)
        for key, child in value.items():
            walk(child, f"{path}.{key}")
    elif isinstance(value, list):
        for index, child in enumerate(value):
            walk(child, f"{path}[{index}]")

walk(safe_diagnostics)

try:
    import pandas as pd
    from IPython.display import display

    if records:
        display(pd.DataFrame(records).fillna(""))
    else:
        print("No status-like records found; inspect the JSON above.")
except Exception as exc:
    print(f"Table display unavailable: {exc}")
    for row in records:
        print(row)


## 4. Run a no-model smoke extraction

This verifies that the current runtime can generate and validate the deterministic threshold demo without optional ML dependencies.


In [ ]:
run([sys.executable, "examples/make_demo_video.py", "--out", "examples/demo_red_ball.mp4"])
run([
    sys.executable,
    "-m",
    "motionjson.cli",
    "extract",
    "examples/demo_red_ball.mp4",
    "--out",
    "out/diagnostics_red_ball",
    "--mask-provider",
    "threshold",
    "--lower-hsv",
    "0,80,80",
    "--upper-hsv",
    "12,255,255",
    "--sample-fps",
    "12",
    "--max-frames",
    "12",
])
run([sys.executable, "-m", "motionjson.cli", "validate", "out/diagnostics_red_ball"])


## 5. Save diagnostics for an issue or bug report


In [ ]:
report_path = REPO_DIR / "motionjson_colab_provider_diagnostics.json"
report_path.write_text(json.dumps(safe_diagnostics, indent=2), encoding="utf-8")
print(f"Wrote {report_path}")
try:
    from google.colab import files  # type: ignore

    files.download(str(report_path))
except Exception as exc:
    print(f"Download helper unavailable outside Colab: {exc}")
